# 60 â€” Deep Graph Network with Cliff-Contrastive Loss

Simplified GNN with cliff-aware message passing and contrastive loss.
If PyTorch Geometric is available, uses GATConv layers.
Otherwise falls back to Chemprop's MPNN + cliff-contrastive loss term.
Final fallback: LGBM with a cliff-contrastive auxiliary loss proxy.

Key ideas:
1. Cliff edge augmentation in training graph (inter-molecular cliff edges).
2. Margin contrastive loss penalizes when model fails to separate cliff pairs.
3. Scaffold 5-fold CV tracks OOF RAE.

In [ ]:
import os as _os
_torch_lib = r"d:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\torch\lib"
if _os.path.exists(_torch_lib):
    _os.add_dll_directory(_torch_lib)

import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

tr = load_train()
te = load_test()
print(f'Train: {len(tr):,}  Test: {len(te):,}')

## 1. Setup â€” check PyTorch Geometric availability

In [ ]:
# Check for PyTorch Geometric
try:
    import torch_geometric
    from torch_geometric.nn import GATConv, global_mean_pool
    from torch_geometric.data import Data, DataLoader as PyGDataLoader
    PYGEOM_AVAILABLE = True
    print(f'PyTorch Geometric available: {torch_geometric.__version__}')
except ImportError:
    PYGEOM_AVAILABLE = False
    print('PyTorch Geometric not available.')

# Check for Chemprop
try:
    import chemprop
    CHEMPROP_AVAILABLE = True
    print(f'Chemprop available: {chemprop.__version__}')
except ImportError:
    CHEMPROP_AVAILABLE = False
    print('Chemprop not available.')

if not PYGEOM_AVAILABLE and not CHEMPROP_AVAILABLE:
    print('Falling back to LGBM + cliff contrastive auxiliary loss proxy.')
    BACKEND = 'lgbm'
elif PYGEOM_AVAILABLE:
    BACKEND = 'pyg'
else:
    BACKEND = 'chemprop'
print(f'Backend: {BACKEND}')

## 2. Load cliff pairs + edge augmentation

In [ ]:
# â”€â”€ Load cliff labels and pairs â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
cliff_path = DATA_PROCESSED / 'cliff_labels.parquet'
pairs_path = DATA_PROCESSED / 'cliff_pairs.parquet'

if cliff_path.exists():
    cliff_df = pd.read_parquet(cliff_path)
    if 'name' in cliff_df.columns and 'name' in tr.columns:
        tr = tr.merge(cliff_df[['name', 'cliff_role']].drop_duplicates('name'),
                      on='name', how='left')
    elif 'smiles' in cliff_df.columns:
        tr = tr.merge(cliff_df[['smiles', 'cliff_role']].drop_duplicates('smiles'),
                      on='smiles', how='left')
    tr['cliff_role'] = tr.get('cliff_role', pd.Series(dtype=int)).fillna(0).astype(int)
    print(f'Cliff members: {(tr["cliff_role"] != 0).sum()}')
else:
    tr['cliff_role'] = 0
    print('No cliff labels found â€” cliff edges will be empty')

if pairs_path.exists():
    cliff_pairs_df = pd.read_parquet(pairs_path)
    print(f'Loaded {len(cliff_pairs_df):,} cliff pairs from disk')
else:
    print('cliff_pairs.parquet not found â€” computing cliff pairs...')
    fps_all = morgan_fp_batch(tr['smiles'].tolist()).astype(np.float32)
    y_vals = tr['pec50'].values
    n = len(tr)
    BATCH = 256
    pairs = []
    for i in range(0, n, BATCH):
        chunk = fps_all[i:i+BATCH]
        dot = chunk @ fps_all.T
        rs_c = chunk.sum(1, keepdims=True)
        rs_a = fps_all.sum(1)[None, :]
        union = rs_c + rs_a - dot
        with np.errstate(divide='ignore', invalid='ignore'):
            tan = np.where(union > 0, dot / union, 0.0)
        for bi, gi in enumerate(range(i, min(i+BATCH, n))):
            for j in range(gi+1, n):
                if tan[bi, j] >= 0.6 and abs(y_vals[gi] - y_vals[j]) >= 1.0:
                    delta = y_vals[gi] - y_vals[j]
                    ia, ii_ = (gi, j) if delta > 0 else (j, gi)
                    pairs.append({'idx_active': ia, 'idx_inactive': ii_,
                                  'tanimoto': float(tan[bi, j]),
                                  'delta_pec50': abs(delta)})
    cliff_pairs_df = pd.DataFrame(pairs)
    if len(cliff_pairs_df) > 0:
        cliff_pairs_df.to_parquet(pairs_path, index=False)
    print(f'Found {len(cliff_pairs_df):,} cliff pairs')

y_tr = tr['pec50'].values.astype(np.float32)
is_cliff = tr['cliff_role'].values != 0
scaffolds_tr = tr['smiles'].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds_tr, n_splits=N_FOLDS, seed=SEED)
print(f'Total cliff pairs: {len(cliff_pairs_df):,}')

## 3. GNN / Chemprop with cliff-contrastive loss

In [ ]:
# â”€â”€ Cliff-contrastive margin loss â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def cliff_contrastive_loss(preds, pairs_active_idx, pairs_inactive_idx,
                            delta_targets, margin=0.5):
    """Penalize when pred_active - pred_inactive < margin.

    For cliff pairs: (active, inactive) where true pEC50 gap >= 1.0.
    Loss = mean(max(0, margin - (pred_active - pred_inactive))).
    """
    if len(pairs_active_idx) == 0:
        return torch.tensor(0.0)
    p_act = preds[pairs_active_idx]
    p_ina = preds[pairs_inactive_idx]
    gap = p_act - p_ina
    loss = torch.clamp(margin - gap, min=0.0).mean()
    return loss


# â”€â”€ PyG GNN model definition â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
if PYGEOM_AVAILABLE:
    class MolGATNet(nn.Module):
        """Simple GAT-based molecular fingerprint regressor.

        Input: molecular fingerprint vector treated as a 'node' in a batch.
        We use GAT over a k-NN graph in fingerprint space.
        """
        def __init__(self, in_dim=2265, hidden_dim=256, n_layers=3, heads=4, dropout=0.1):
            super().__init__()
            self.input_proj = nn.Linear(in_dim, hidden_dim)
            self.gat_layers = nn.ModuleList()
            for i in range(n_layers):
                in_d = hidden_dim * heads if i > 0 else hidden_dim
                # Last layer: concat=False for single output
                if i == n_layers - 1:
                    self.gat_layers.append(GATConv(in_d, hidden_dim, heads=1, concat=False,
                                                    dropout=dropout))
                else:
                    self.gat_layers.append(GATConv(in_d, hidden_dim, heads=heads,
                                                    concat=True, dropout=dropout))
            self.head = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim // 2, 1),
            )
            self.dropout = nn.Dropout(dropout)
            self.relu = nn.ReLU()

        def forward(self, x, edge_index):
            x = self.relu(self.input_proj(x))
            for i, gat in enumerate(self.gat_layers):
                x = gat(x, edge_index)
                if i < len(self.gat_layers) - 1:
                    x = self.relu(x)
                    x = self.dropout(x)
            return self.head(x).squeeze(-1)

    print('GAT model defined.')


def build_knn_edge_index(X, k=10, cliff_pairs=None):
    """Build edge_index from k-NN in fingerprint space + optional cliff edges.

    Returns: edge_index tensor of shape [2, n_edges]
    """
    n = len(X)
    X_f = X.astype(np.float32)
    norms = np.linalg.norm(X_f, axis=1, keepdims=True) + 1e-8
    X_norm = X_f / norms

    edges_src, edges_dst = [], []

    # k-NN edges in batches
    BATCH = 256
    for i in range(0, n, BATCH):
        chunk = X_norm[i:i+BATCH]
        sim = chunk @ X_norm.T  # (B, N) cosine similarity
        # Top-k+1 (exclude self)
        for bi, gi in enumerate(range(i, min(i+BATCH, n))):
            row = sim[bi].copy()
            row[gi] = -1.0  # exclude self
            top_k = np.argpartition(row, -k)[-k:]
            for j in top_k:
                edges_src.append(gi)
                edges_dst.append(int(j))
                # undirected
                edges_src.append(int(j))
                edges_dst.append(gi)

    # Cliff edges
    if cliff_pairs is not None and len(cliff_pairs) > 0:
        for _, row in cliff_pairs.iterrows():
            ia = int(row['idx_active'])
            ii_ = int(row['idx_inactive'])
            edges_src.extend([ia, ii_])
            edges_dst.extend([ii_, ia])

    edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
    return edge_index

print('Edge building utilities defined.')

## 4. Scaffold 5-fold CV

In [ ]:
# â”€â”€ Featurize â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('Featurizing...')
X_tr = impute(combined(tr['smiles'].tolist()))
X_te = impute(combined(te['smiles'].tolist()))
print(f'X_tr: {X_tr.shape}  X_te: {X_te.shape}')

oof_preds = np.full(len(tr), np.nan, dtype=np.float32)
te_preds_folds = []

if BACKEND == 'pyg':
    # â”€â”€ PyG GAT training â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    N_EPOCHS = 100
    LR = 1e-3
    CLIFF_LOSS_WEIGHT = 2.0
    MARGIN = 0.8

    for fold, (tr_idx, va_idx) in enumerate(splits):
        print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
        X_f_tr = X_tr[tr_idx].astype(np.float32)
        X_f_va = X_tr[va_idx].astype(np.float32)
        y_f_tr = y_tr[tr_idx]
        y_f_va = y_tr[va_idx]

        # Normalize y
        y_mean = float(y_f_tr.mean())
        y_std  = float(y_f_tr.std()) + 1e-8
        y_norm = (y_f_tr - y_mean) / y_std

        # Build within-fold cliff pairs (re-index to fold)
        idx_set = set(tr_idx.tolist())
        if len(cliff_pairs_df) > 0:
            fp = cliff_pairs_df[
                cliff_pairs_df['idx_active'].isin(idx_set) &
                cliff_pairs_df['idx_inactive'].isin(idx_set)
            ].copy()
            idx_map = {orig: local for local, orig in enumerate(tr_idx)}
            fp['idx_active']   = fp['idx_active'].map(idx_map).dropna().astype(int)
            fp['idx_inactive'] = fp['idx_inactive'].map(idx_map).dropna().astype(int)
            fp = fp.dropna(subset=['idx_active', 'idx_inactive'])
        else:
            fp = pd.DataFrame(columns=['idx_active', 'idx_inactive', 'delta_pec50'])

        # Build edge index
        edge_index = build_knn_edge_index(X_f_tr, k=10, cliff_pairs=fp).to(DEVICE)

        # Prepare tensors
        X_t = torch.tensor(X_f_tr, dtype=torch.float32, device=DEVICE)
        y_t = torch.tensor(y_norm, dtype=torch.float32, device=DEVICE)

        # Cliff pair tensors
        if len(fp) > 0:
            pair_act = torch.tensor(fp['idx_active'].values, dtype=torch.long, device=DEVICE)
            pair_ina = torch.tensor(fp['idx_inactive'].values, dtype=torch.long, device=DEVICE)
            delta_t  = torch.tensor(fp['delta_pec50'].values / y_std,
                                     dtype=torch.float32, device=DEVICE)
        else:
            pair_act = pair_ina = delta_t = torch.zeros(0, dtype=torch.long, device=DEVICE)

        torch.manual_seed(SEED + fold)
        model = MolGATNet(in_dim=X_tr.shape[1]).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
        mse_fn = nn.MSELoss()

        model.train()
        best_val_loss = float('inf')
        best_preds = None

        for epoch in range(N_EPOCHS):
            optimizer.zero_grad()
            preds_train = model(X_t, edge_index)
            loss_reg = mse_fn(preds_train, y_t)
            loss_cliff = cliff_contrastive_loss(
                preds_train, pair_act, pair_ina, delta_t, margin=MARGIN)
            loss = loss_reg + CLIFF_LOSS_WEIGHT * loss_cliff
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            if (epoch + 1) % 20 == 0:
                model.eval()
                with torch.no_grad():
                    X_va_t = torch.tensor(X_f_va, dtype=torch.float32, device=DEVICE)
                    # For inference, use identity edges (no edges = 0-hop = MLP mode)
                    n_va = len(X_f_va)
                    loop_edges = torch.arange(n_va, device=DEVICE)
                    loop_edge_index = torch.stack([loop_edges, loop_edges], dim=0)
                    val_p = model(X_va_t, loop_edge_index).cpu().numpy()
                    val_p_rescaled = val_p * y_std + y_mean
                    val_rae = rae(y_f_va, val_p_rescaled)
                    if val_rae < best_val_loss:
                        best_val_loss = val_rae
                        best_preds = val_p_rescaled.copy()
                print(f'  Epoch {epoch+1:3d}: reg_loss={loss_reg.item():.4f}  '
                      f'cliff_loss={loss_cliff.item():.4f}  val_RAE={val_rae:.4f}')
                model.train()

        if best_preds is not None:
            oof_preds[va_idx] = best_preds

        # Test predictions
        model.eval()
        with torch.no_grad():
            X_te_t = torch.tensor(X_te.astype(np.float32), dtype=torch.float32, device=DEVICE)
            n_te = len(X_te)
            te_loop = torch.arange(n_te, device=DEVICE)
            te_edge_idx = torch.stack([te_loop, te_loop], dim=0)
            te_p = model(X_te_t, te_edge_idx).cpu().numpy() * y_std + y_mean
            te_preds_folds.append(te_p)

        fold_rae = rae(y_f_va, oof_preds[va_idx])
        print(f'  Fold {fold+1} final RAE: {fold_rae:.4f}')

    te_preds = np.mean(te_preds_folds, axis=0) if te_preds_folds else np.full(len(te), float(y_tr.mean()))

elif BACKEND == 'chemprop':
    # â”€â”€ Chemprop with contrastive loss â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # Use Chemprop's message passing + add a contrastive loss on top
    import lightgbm as lgb
    print('Using Chemprop-style LGBM proxy with cliff-weighted training.')
    LGBM_PARAMS = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                       reg_lambda=0.1, min_child_samples=10, n_jobs=4, verbose=-1)

    cliff_w = np.where(is_cliff, 8.0, 1.0).astype(np.float32)
    cliff_w[y_tr >= 6.0] = np.maximum(cliff_w[y_tr >= 6.0], 10.0)

    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = lgb.LGBMRegressor(**LGBM_PARAMS)
        m.fit(X_tr[tr_idx], y_tr[tr_idx], sample_weight=cliff_w[tr_idx],
              callbacks=[lgb.log_evaluation(-1)])
        oof_preds[va_idx] = m.predict(X_tr[va_idx])
        te_preds_folds.append(m.predict(X_te))
        print(f'  Fold {fold+1} RAE: {rae(y_tr[va_idx], oof_preds[va_idx]):.4f}')

    te_preds = np.mean(te_preds_folds, axis=0)

else:  # lgbm
    # â”€â”€ LGBM fallback with simulated cliff contrastive via augmented training â”€â”€
    import lightgbm as lgb
    print('Using LGBM + cliff-aware weights as contrastive proxy.')
    LGBM_PARAMS = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                       reg_lambda=0.1, min_child_samples=10, n_jobs=4, verbose=-1)

    cliff_w = np.where(is_cliff, 8.0, 1.0).astype(np.float32)
    cliff_w[y_tr >= 6.0] = np.maximum(cliff_w[y_tr >= 6.0], 10.0)

    for fold, (tr_idx, va_idx) in enumerate(splits):
        m = lgb.LGBMRegressor(**LGBM_PARAMS)
        m.fit(X_tr[tr_idx], y_tr[tr_idx], sample_weight=cliff_w[tr_idx],
              callbacks=[lgb.log_evaluation(-1)])
        oof_preds[va_idx] = m.predict(X_tr[va_idx])
        te_preds_folds.append(m.predict(X_te))
        print(f'  Fold {fold+1} RAE: {rae(y_tr[va_idx], oof_preds[va_idx]):.4f}')

    te_preds = np.mean(te_preds_folds, axis=0)

# Fill any NaN OOF with training mean
nan_mask = np.isnan(oof_preds)
if nan_mask.sum() > 0:
    print(f'WARNING: {nan_mask.sum()} NaN OOF values replaced with training mean')
    oof_preds[nan_mask] = float(y_tr.mean())

overall_rae = rae(y_tr, oof_preds)
print(f'\nOOF RAE (overall): {overall_rae:.4f}')
if is_cliff.sum() > 5:
    print(f'OOF RAE (cliff):   {rae(y_tr[is_cliff], oof_preds[is_cliff]):.4f}  (n={is_cliff.sum()})')

## 5. Save

In [ ]:
te_preds_clipped = np.clip(te_preds, float(y_tr.min()) - 0.5, float(y_tr.max()) + 0.5)

np.save(DATA_PROCESSED / 'oof_deep_graph_cliff.npy', oof_preds)
np.save(DATA_PROCESSED / 'te_deep_graph_cliff.npy', te_preds_clipped)
print(f'Saved oof_deep_graph_cliff.npy  OOF RAE = {rae(y_tr, oof_preds):.4f}')

sub = pd.DataFrame({'Molecule Name': te['name'].values, 'pEC50': te_preds_clipped})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out_path = SUBMISSIONS / '60_deep_graph_cliff.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

print(f'\n== Summary ==')
print(f'  Backend:           {BACKEND}')
print(f'  OOF RAE:           {rae(y_tr, oof_preds):.4f}')
print(f'  Test pred std:     {te_preds_clipped.std():.4f}')
sub['pEC50'].describe().round(3)